# S08 : One-Step reaction prediction

Authors:
- [Tieu-Long Phan](https://tieulongphan.github.io/), Leipzig University - Southern Denmark University


## Aim of this talktorial

## Learning outcomes


## Roadmap


## 0. Setup & data

In [2]:
import rdkit
import pandas as pd
import networkx as nx
from pathlib import Path
import importlib.metadata as m
from synkit.IO import load_database

print("RDKit version:", rdkit.__version__)
print("NetworkX version:", nx.__version__)
print("SynKit version:", m.version('synkit'))

DATA_DIR = Path("data")
DATA_PATH = DATA_DIR / "smart.json.gz"
data = pd.DataFrame(
    load_database(DATA_PATH)[:10000]
)  # we use 10000 data points for fast process
display(data.head())
print(data.shape)

RDKit version: 2025.09.3
NetworkX version: 3.6.1
SynKit version: 1.1.0


,R-id,smart
0,7873,[CH3:1][C:2]([CH3:3])([CH3:4])[O:5][C:6](=[O:7...
1,42468,[Br:1][CH2:2][c:3]1[cH:4][cH:6][c:7]([Br:8])[c...
2,24541,[CH3:1][N:2]([CH3:3])[CH2:4][CH2:5][NH:6][H:7]...
3,40699,[CH3:1][O:2][c:3]1[cH:4][cH:6][cH:7][cH:8][c:5...
4,34914,[CH3:1][CH:2]([CH3:3])[O:4][H:5].[O:6]=[C:7]([...


(10000, 2)


## 1. Forward prediction

## 1. Theory: one-step rewriting in both directions

A DPO rule is a span of typed graphs
\[
p:\quad L \xleftarrow{\,l\,} K \xrightarrow{\,r\,} R
\]
where:
- \(L\) is the **pre-condition** (what must be present to react),
- \(R\) is the **post-condition** (what is produced),
- \(K\) is the **preserved interface** (what stays the same).

Given a host graph \(G\) (a set of reactant molecules as a disjoint union),
a **match** is an injective morphism
\[
m: L \hookrightarrow G
\]
that respects atom and bond labels.

### Forward (synthesis)
The DPO construction removes the part \(L\setminus K\) and glues in \(R\setminus K\), producing \(G'\).

### Backward (retrosynthesis)
The inverse rule is simply
\[
p^{-1}:\quad R \xleftarrow{\,r\,} K \xrightarrow{\,l\,} L
\]
and applying \(p^{-1}\) to a product graph generates plausible precursors.

**Chemistry meaning:** a rule extracted from atom-mapped data can be used in either direction as a
one-step generator; directionality is controlled by which side you treat as “host”.


In [ ]:
# --- Canonicalization (from S05, minimal subset) ---


def clear_atom_maps(m: Chem.Mol) -> Chem.Mol:
    mm = Chem.Mol(m)
    for a in mm.GetAtoms():
        a.SetAtomMapNum(0)
    return mm


def mol_key_unmapped(m: Chem.Mol) -> str:
    return Chem.MolToSmiles(clear_atom_maps(m), canonical=True)


def canonical_ranks_unmapped(m: Chem.Mol) -> List[int]:
    mm = clear_atom_maps(m)
    return list(Chem.CanonicalRankAtoms(mm, includeChirality=True))


def mapped_atoms_in_mol(m: Chem.Mol) -> List[Tuple[int, int]]:
    out = []
    for a in m.GetAtoms():
        mid = int(a.GetAtomMapNum() or 0)
        if mid > 0:
            out.append((a.GetIdx(), mid))
    return out


def canonical_map_renaming(mapped_rxn: str) -> Dict[int, int]:
    r_smis, _agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]

    def collect(mols: List[Chem.Mol], side: int):
        items = []
        for m in sorted(mols, key=mol_key_unmapped):
            key_m = mol_key_unmapped(m)
            ranks = canonical_ranks_unmapped(m)
            for aidx, mid in mapped_atoms_in_mol(m):
                a = m.GetAtomWithIdx(aidx)
                k = (
                    side,
                    key_m,
                    int(ranks[aidx]),
                    a.GetSymbol(),
                    int(a.GetFormalCharge()),
                    bool(a.GetIsAromatic()),
                )
                items.append((k, mid))
        return items

    seen = {}
    for k, mid in collect(r_mols, 0):
        seen.setdefault(mid, k)
    for k, mid in collect(p_mols, 1):
        seen.setdefault(mid, k)

    ordered = sorted([(k, mid) for mid, k in seen.items()])
    return {mid: i + 1 for i, (k, mid) in enumerate(ordered)}


def apply_map_renaming_to_rxn(mapped_rxn: str, ren: Dict[int, int]) -> str:
    r_smis, agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]
    a_mols = [Chem.MolFromSmiles(s) for s in agents] if agents else []

    for m in r_mols + p_mols + a_mols:
        for a in m.GetAtoms():
            mid = int(a.GetAtomMapNum() or 0)
            if mid > 0 and mid in ren:
                a.SetAtomMapNum(int(ren[mid]))

    def ms(m):
        return Chem.MolToSmiles(m, canonical=True)

    r_out = ".".join(ms(m) for m in sorted(r_mols, key=mol_key_unmapped))
    p_out = ".".join(ms(m) for m in sorted(p_mols, key=mol_key_unmapped))
    if agents:
        a_out = ".".join(ms(m) for m in sorted(a_mols, key=mol_key_unmapped))
        return f"{r_out}>{a_out}>{p_out}"
    return f"{r_out}>>{p_out}"


def canonicalize_mapped_rxn(mapped_rxn: str) -> str:
    return apply_map_renaming_to_rxn(mapped_rxn, canonical_map_renaming(mapped_rxn))


def unmap_smiles(sm: str) -> str:
    m = Chem.MolFromSmiles(sm)
    if m is None:
        raise ValueError(f"Bad SMILES: {sm}")
    return Chem.MolToSmiles(clear_atom_maps(m), canonical=True)


def rxn_sides_unmapped(mapped_rxn: str) -> Tuple[List[str], List[str]]:
    r_smis, _agents, p_smis = split_rxn_smiles(mapped_rxn)
    r = [unmap_smiles(s) for s in r_smis]
    p = [unmap_smiles(s) for s in p_smis]
    return r, p


def mixture_from_smiles_list(smis: List[str]) -> str:
    return ".".join(sorted(smis))

In [ ]:
from dataclasses import dataclass
from collections import deque


def bond_order_rdkit(b: Chem.Bond) -> float:
    bt = b.GetBondType()
    if bt == Chem.rdchem.BondType.SINGLE:
        return 1.0
    if bt == Chem.rdchem.BondType.DOUBLE:
        return 2.0
    if bt == Chem.rdchem.BondType.TRIPLE:
        return 3.0
    if bt == Chem.rdchem.BondType.AROMATIC:
        return 1.5
    return float(b.GetBondTypeAsDouble())


def atoms_by_map_id(mols: List[Chem.Mol]) -> Dict[int, Chem.Atom]:
    out = {}
    for m in mols:
        for a in m.GetAtoms():
            mid = int(a.GetAtomMapNum() or 0)
            if mid > 0:
                out[mid] = a
    return out


def bonds_by_map_pair(mols: List[Chem.Mol]) -> Dict[Tuple[int, int], float]:
    out = {}
    for m in mols:
        for b in m.GetBonds():
            ai = int(b.GetBeginAtom().GetAtomMapNum() or 0)
            aj = int(b.GetEndAtom().GetAtomMapNum() or 0)
            if ai <= 0 or aj <= 0:
                continue
            u, v = (ai, aj) if ai < aj else (aj, ai)
            out[(u, v)] = bond_order_rdkit(b)
    return out


def mapped_rxn_to_its(mapped_rxn: str) -> nx.Graph:
    r_smis, _agents, p_smis = split_rxn_smiles(mapped_rxn)
    r_mols = [Chem.MolFromSmiles(s) for s in r_smis]
    p_mols = [Chem.MolFromSmiles(s) for s in p_smis]
    if any(m is None for m in r_mols + p_mols):
        raise ValueError("Bad SMILES in mapped reaction.")
    r_atoms = atoms_by_map_id(r_mols)
    p_atoms = atoms_by_map_id(p_mols)
    V = set(r_atoms) | set(p_atoms)
    G = nx.Graph()
    for mid in sorted(V):
        ar = r_atoms.get(mid)
        ap = p_atoms.get(mid)
        a_any = ar if ar is not None else ap
        G.add_node(
            mid,
            symbol=a_any.GetSymbol(),
            aromatic=bool(a_any.GetIsAromatic()),
            r_charge=(int(ar.GetFormalCharge()) if ar is not None else None),
            p_charge=(int(ap.GetFormalCharge()) if ap is not None else None),
        )
    rb = bonds_by_map_pair(r_mols)
    pb = bonds_by_map_pair(p_mols)
    for u, v in sorted(set(rb) | set(pb)):
        ro = rb.get((u, v))
        po = pb.get((u, v))
        G.add_edge(u, v, r_order=ro, p_order=po, changed=(ro != po))
    return G


def reaction_center_core_nodes(
    G: nx.Graph, include_charge_changes: bool = True
) -> set[int]:
    C = set()
    for u, v, d in G.edges(data=True):
        if d.get("r_order") != d.get("p_order"):
            C.add(u)
            C.add(v)
    if include_charge_changes:
        for v, nd in G.nodes(data=True):
            rc, pc = nd.get("r_charge"), nd.get("p_charge")
            if (rc is not None) and (pc is not None) and (rc != pc):
                C.add(v)
    return C


def expand_radius(G: nx.Graph, core: set[int], radius: int) -> set[int]:
    if radius <= 0:
        return set(core)
    seen = set(core)
    q = deque([(v, 0) for v in core])
    while q:
        v, d = q.popleft()
        if d >= radius:
            continue
        for u in G.neighbors(v):
            if u not in seen:
                seen.add(u)
                q.append((u, d + 1))
    return seen


@dataclass(frozen=True)
class DPORule:
    L: nx.Graph
    K: nx.Graph
    R: nx.Graph
    meta: Dict[str, object]


def its_to_dpo_rule(its: nx.Graph, core_radius: int = 1) -> DPORule:
    core = reaction_center_core_nodes(its, include_charge_changes=True)
    C = expand_radius(its, core, radius=core_radius)

    L_nodes = [v for v in C if its.nodes[v].get("r_charge") is not None]
    R_nodes = [v for v in C if its.nodes[v].get("p_charge") is not None]
    K_nodes = sorted(set(L_nodes) & set(R_nodes))

    L = nx.Graph()
    R = nx.Graph()
    K = nx.Graph()

    for v in sorted(set(L_nodes) | set(R_nodes)):
        nd = its.nodes[v]
        base = dict(symbol=nd["symbol"], aromatic=nd["aromatic"])
        if v in L_nodes:
            L.add_node(v, **base, formal_charge=int(nd["r_charge"] or 0))
        if v in R_nodes:
            R.add_node(v, **base, formal_charge=int(nd["p_charge"] or 0))
        if v in K_nodes:
            K.add_node(v, **base, formal_charge=int(nd["r_charge"] or 0))

    for u, v, d in its.edges(data=True):
        ro, po = d.get("r_order"), d.get("p_order")
        if (ro is not None) and (u in L) and (v in L):
            L.add_edge(u, v, order=float(ro))
        if (po is not None) and (u in R) and (v in R):
            R.add_edge(u, v, order=float(po))
        if (
            (ro is not None)
            and (po is not None)
            and (ro == po)
            and (u in K)
            and (v in K)
        ):
            K.add_edge(u, v, order=float(ro))

    meta = {"core_nodes": sorted(core), "context_radius": int(core_radius)}
    return DPORule(L=L, K=K, R=R, meta=meta)

In [ ]:
# --- RDKit <-> typed graph utilities (float bond order, aromatic=1.5) ---


def mol_to_graph(mol: Chem.Mol, include_implicit_h: bool = True) -> nx.Graph:
    """
    RDKit Mol -> typed NetworkX graph.

    Node attrs:
      - symbol, formal_charge, aromatic
      - total_h (optional)

    Edge attrs:
      - order (float; aromatic=1.5)
    """
    G = nx.Graph()
    for a in mol.GetAtoms():
        i = a.GetIdx()
        attrs: Dict[str, object] = {
            "symbol": a.GetSymbol(),
            "formal_charge": int(a.GetFormalCharge()),
            "aromatic": bool(a.GetIsAromatic()),
        }
        if include_implicit_h:
            attrs["total_h"] = int(a.GetTotalNumHs())
        G.add_node(i, **attrs)

    for b in mol.GetBonds():
        u, v = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bt = b.GetBondType()
        if bt == Chem.rdchem.BondType.AROMATIC:
            order = 1.5
        else:
            order = float(b.GetBondTypeAsDouble())
        G.add_edge(u, v, order=float(order))
    return G


def _bondtype_from_order(order: float) -> Chem.rdchem.BondType:
    if abs(order - 1.5) < 1e-8:
        return Chem.rdchem.BondType.AROMATIC
    if abs(order - 1.0) < 1e-8:
        return Chem.rdchem.BondType.SINGLE
    if abs(order - 2.0) < 1e-8:
        return Chem.rdchem.BondType.DOUBLE
    if abs(order - 3.0) < 1e-8:
        return Chem.rdchem.BondType.TRIPLE
    # fallback
    return Chem.rdchem.BondType.SINGLE


def graph_to_mol(G: nx.Graph) -> Chem.Mol:
    """
    Typed molecular graph -> RDKit Mol (best-effort).

    Supports aromatic bonds via order=1.5.
    """
    rw = Chem.RWMol()
    nx_to_rdk: Dict[int, int] = {}

    for node in sorted(G.nodes()):
        d = G.nodes[node]
        atom = Chem.Atom(str(d.get("symbol", "C")))
        atom.SetFormalCharge(int(d.get("formal_charge", 0)))
        atom.SetIsAromatic(bool(d.get("aromatic", False)))
        nx_to_rdk[node] = rw.AddAtom(atom)

    for u, v, d in G.edges(data=True):
        order = float(d.get("order", 1.0))
        btype = _bondtype_from_order(order)
        rw.AddBond(nx_to_rdk[u], nx_to_rdk[v], btype)
        if btype == Chem.rdchem.BondType.AROMATIC:
            # ensure aromatic flags are set on atoms (RDKit sometimes needs this)
            rw.GetAtomWithIdx(nx_to_rdk[u]).SetIsAromatic(True)
            rw.GetAtomWithIdx(nx_to_rdk[v]).SetIsAromatic(True)

    mol = rw.GetMol()
    Chem.SanitizeMol(mol)
    return mol


def mixture_smiles(m: Chem.Mol) -> str:
    """Canonical dot-separated SMILES for a possibly disconnected molecule."""
    frags = Chem.GetMolFrags(m, asMols=True, sanitizeFrags=True)
    smis = [Chem.MolToSmiles(f, canonical=True) for f in frags]
    return ".".join(sorted(smis))


def smiles_list_to_host_graph(smis: List[str]) -> nx.Graph:
    """Disjoint union of multiple molecules as one graph with offset node ids."""
    G = nx.Graph()
    offset = 0
    for sm in smis:
        m = Chem.MolFromSmiles(sm)
        if m is None:
            raise ValueError(f"Bad SMILES: {sm}")
        Gi = mol_to_graph(m)
        mapping = {i: i + offset for i in Gi.nodes()}
        Gi = nx.relabel_nodes(Gi, mapping, copy=True)
        G = nx.compose(G, Gi)
        offset += m.GetNumAtoms()
    return G

In [ ]:
# --- DPO matching and one-step application (product graph) ---


def node_match(keys: Sequence[str] = ("symbol", "formal_charge", "aromatic")):
    return iso.categorical_node_match(list(keys), [None] * len(keys))


def edge_match(u: Dict[str, object], v: Dict[str, object]) -> bool:
    return float(u.get("order", 0.0)) == float(v.get("order", 0.0))


def dpo_match_iter(host: nx.Graph, L: nx.Graph) -> Iterable[Dict[Hashable, Hashable]]:
    """
    Yield injective matches m: V(L) -> V(host).
    Implemented via NetworkX subgraph isomorphism.
    """
    GM = iso.GraphMatcher(host, L, node_match=node_match(), edge_match=edge_match)
    # GraphMatcher returns mappings host_node -> pattern_node; invert
    for mp in GM.subgraph_isomorphisms_iter():
        yield {ln: hn for hn, ln in mp.items()}


def invert_rule(rule: DPORule) -> DPORule:
    """Return inverse rule R <- K -> L (meta preserved with a flag)."""
    meta = dict(rule.meta)
    meta["inverted"] = True
    return DPORule(L=rule.R, K=rule.K, R=rule.L, meta=meta)


def apply_dpo_once(
    host: nx.Graph, rule: DPORule, m: Dict[Hashable, Hashable]
) -> nx.Graph:
    """
    Apply a DPO rule to a host graph using a chosen match m: L -> host.
    Returns a new product graph host'.
    """
    L, K, R = rule.L, rule.K, rule.R
    host2 = host.copy()

    L_nodes = set(L.nodes())
    K_nodes = set(K.nodes())
    R_nodes = set(R.nodes())

    # 1) delete nodes in L\K
    del_rule_nodes = L_nodes - K_nodes
    del_host_nodes = {m[x] for x in del_rule_nodes}

    for hn in del_host_nodes:
        if hn in host2:
            host2.remove_node(hn)

    # 2) delete edges corresponding to (L edges not in K edges)
    #    (changed bonds are modeled as delete+add because they are not in K)
    def norm_edge(u, v):
        return (u, v) if u <= v else (v, u)

    K_edge_endpoints = {norm_edge(u, v) for u, v in K.edges()}
    for u, v in L.edges():
        if norm_edge(u, v) in K_edge_endpoints:
            continue
        hu, hv = m[u], m[v]
        if host2.has_edge(hu, hv):
            host2.remove_edge(hu, hv)

    # 3) create fresh nodes for R\K
    existing = [n for n in host2.nodes() if isinstance(n, int)]
    next_id = (max(existing) + 1) if existing else 0
    fresh: Dict[Hashable, int] = {}
    for rn in R_nodes - K_nodes:
        fresh[rn] = next_id
        next_id += 1
        host2.add_node(fresh[rn], **R.nodes[rn])

    def host_id(rn: Hashable) -> int:
        return int(m[rn]) if rn in K_nodes else int(fresh[rn])

    # 4) add edges in R that are not in K
    for u, v, d in R.edges(data=True):
        if norm_edge(u, v) in K_edge_endpoints:
            continue
        hu, hv = host_id(u), host_id(v)
        host2.add_edge(hu, hv, **d)

    return host2


def apply_rule(
    host: nx.Graph,
    rule: DPORule,
    deduplicate: bool = True,
    max_matches: Optional[int] = 50,
) -> List[nx.Graph]:
    """Enumerate matches and apply the rule; return product graphs."""
    out: List[nx.Graph] = []
    for i, m in enumerate(dpo_match_iter(host, rule.L)):
        if max_matches is not None and i >= max_matches:
            break
        try:
            out.append(apply_dpo_once(host, rule, m))
        except Exception:
            continue

    if not deduplicate:
        return out

    uniq: List[nx.Graph] = []
    seen: set[str] = set()
    for Gp in out:
        try:
            sm = mixture_smiles(graph_to_mol(Gp))
        except Exception:
            continue
        if sm not in seen:
            seen.add(sm)
            uniq.append(Gp)
    return uniq


def apply_rule_smiles(
    reactant_smis: List[str],
    rule: DPORule,
    direction: str = "forward",
) -> List[str]:
    """
    Convenience wrapper: reactant SMILES list -> candidate product mixture SMILES.
    direction: "forward" uses rule, "backward" uses inverse rule.
    """
    host = smiles_list_to_host_graph(reactant_smis)
    rr = rule if direction == "forward" else invert_rule(rule)
    outs = apply_rule(host, rr, deduplicate=True)
    smis: List[str] = []
    for Gp in outs:
        try:
            smis.append(mixture_smiles(graph_to_mol(Gp)))
        except Exception:
            continue
    return sorted(set(smis))

In [ ]:
# --- Demo: extract a rule from one reaction, apply forward and backward ---
rxn0 = df["mapped_rxn"].dropna().iloc[0]
rxn0_can = canonicalize_mapped_rxn(rxn0)
r0_smis, p0_smis = rxn_sides_unmapped(rxn0_can)

print("Canonicalized mapped reaction (for rule extraction):")
print(rxn0_can)
print("\nUnmapped reactants:", r0_smis)
print("Unmapped products :", p0_smis)

rule0 = its_to_dpo_rule(mapped_rxn_to_its(rxn0_can), core_radius=1)

# Forward: reactants -> candidates
cand_p = apply_rule_smiles(r0_smis, rule0, direction="forward")
print("\nForward candidates:")
for s in cand_p[:10]:
    print("  ", s)

target_p = mixture_from_smiles_list(p0_smis)
print("\nTarget product mixture:", target_p)
print("Target in candidates?", target_p in cand_p)

# Backward: products -> candidates (retrosynthesis)
cand_r = apply_rule_smiles(p0_smis, rule0, direction="backward")
print("\nBackward candidates (precursors):")
for s in cand_r[:10]:
    print("  ", s)

target_r = mixture_from_smiles_list(r0_smis)
print("\nTarget reactant mixture:", target_r)
print("Target in backward candidates?", target_r in cand_r)

In [ ]:
from time import perf_counter


def self_reproduction_study(
    df: pd.DataFrame, N: int = 100, core_radius: int = 1
) -> pd.DataFrame:
    rows = []
    t0 = perf_counter()
    for rxn in df["mapped_rxn"].dropna().iloc[:N]:
        try:
            rxn_can = canonicalize_mapped_rxn(rxn)
            r_smis, p_smis = rxn_sides_unmapped(rxn_can)
            target_p = mixture_from_smiles_list(p_smis)
            target_r = mixture_from_smiles_list(r_smis)

            rule = its_to_dpo_rule(mapped_rxn_to_its(rxn_can), core_radius=core_radius)

            cand_p = apply_rule_smiles(r_smis, rule, direction="forward")
            cand_r = apply_rule_smiles(p_smis, rule, direction="backward")

            rows.append(
                {
                    "forward_hit": target_p in cand_p,
                    "backward_hit": target_r in cand_r,
                    "n_forward_cand": len(cand_p),
                    "n_backward_cand": len(cand_r),
                }
            )
        except Exception:
            continue
    dt = perf_counter() - t0
    out = pd.DataFrame(rows)
    out.attrs["runtime_sec"] = dt
    return out


eval_df = self_reproduction_study(df, N=100, core_radius=1)
print("Evaluated:", len(eval_df), "reactions")
print("Runtime (s):", eval_df.attrs.get("runtime_sec"))

if len(eval_df):
    print("Forward hit rate :", eval_df["forward_hit"].mean())
    print("Backward hit rate:", eval_df["backward_hit"].mean())
    print("Avg # forward candidates :", eval_df["n_forward_cand"].mean())
    print("Avg # backward candidates:", eval_df["n_backward_cand"].mean())

    display(eval_df.head(10))

## 3. Discussion

A self-reproduction study (rule extracted from a reaction, then applied back to its own reactants/products)
is a **sanity check**, not a benchmark:

- If the hit rate is low, it usually indicates one of:
  - parsing/mapping inconsistencies in the dataset,
  - missing atoms (unbalanced reactions),
  - overly strict node/edge matching predicates,
  - stereochemistry or aromaticity handling issues.

In S07 we will move from “self reproduction” to **library-level evaluation**:
apply a *set* of rules to each host and study precision/branching trade-offs.


## 4. Exercises

1. **Strictness toggle.** Modify `node_match()` to include/exclude `formal_charge` or `aromatic`.
   How does forward/backward hit rate change?
2. **Context radius.** Repeat `self_reproduction_study` for `core_radius=0,1,2`.
   Does a larger context reduce spurious candidates?
3. **Match explosion.** Find a reaction where `n_forward_cand` is large.
   Inspect the matches and identify a symmetry source.


## Quiz

This section will be expanded in a future revision.
